In [1]:
import pandas as pd

# df1 = pd.read_csv("../../data/dataset.csv", sep="\\|\\|\\|", engine="python").drop(["modern_prompt", "translation_prompt"], axis=1)
df1 = pd.read_csv("../../data/cleaned_dataset.csv", sep="|", engine="python")       # les textes
df2_init = pd.read_csv("../../data/paires_mot_lemme.csv", )                         # les mots
df2 = df2_init.rename(columns={'mot': 'old_french', 'lemme': 'modern'})

In [2]:
df1["type"] = "phrase"
df1.head()


,modern,old_french,type
0,"salut tout le monde, c'est victor. je repensai...","seigneurs et dames, je vous salue, c'est victo...",phrase
1,"salut tout le monde ! aujourd'hui, c'était un ...","saluz tout le monde ! en ce jour, fu moult bon...",phrase
2,"chère sophie, tu ne devineras jamais ce qui m'...","chère sophie, tu ne devineras ja mie ce qui m'...",phrase
3,"bonjour à tous, ici marcel dupré, artisan poti...","bien le bon jour à tous, céans marcel dupré, o...",phrase
4,"yo, c’est lila. alors voilà, l’autre jour, j’é...","salut, c'est lila. lors, l'autre jour, j'estoi...",phrase


In [3]:
df2["type"] = "mot"
df2.head()


,old_french,modern,type
0,Cil,cil,mot
1,qui,qui1,mot
2,fist,faire,mot
3,d',de,mot
4,Erec,Erec,mot


In [4]:
df = pd.concat([df1, df2], ignore_index=True)
df.shape

(33848, 3)

In [5]:
# Séparer phrases et mots pour traitement différencié

sentences_df = df[df['type'] == 'phrase']  # 1000 lignes
words_df = df[df['type'] == 'mot']          # 30000 lignes

print(sentences_df.shape)
print(words_df.shape)

(970, 3)
(32878, 3)


# modelisation

In [6]:
# Division pour les phrases (plus précieuses)
from sklearn.model_selection import train_test_split


train_sentences, temp_sentences = train_test_split(sentences_df, test_size=0.2, random_state=42)
val_sentences, test_sentences = train_test_split(temp_sentences, test_size=0.5, random_state=42)

# Division pour les mots (plus nombreux)
train_words, temp_words = train_test_split(words_df, test_size=0.15, random_state=42)
val_words, test_words = train_test_split(temp_words, test_size=0.5, random_state=42)

print(f"Train: {len(train_sentences)} phrases, {len(train_words)} mots")


Train: 776 phrases, 27946 mots


In [7]:
# Initialiser mT5-small

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_name = "google/mt5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Optimisation mémoire
model.gradient_checkpointing_enable()


from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)


/home/malek/BRIEFS DEV IA/14.NLP/EULA-vaaag-/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/home/malek/BRIEFS DEV IA/14.NLP/EULA-vaaag-/.venv/lib/python3.11/site-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not i

In [8]:
from torch.utils.data import Dataset

class StratifiedOldFrenchDataset(Dataset):
    def __init__(self, sentences_df, words_df, sentence_weight=3.0, max_length=64):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.data = []
        for _, row in sentences_df.iterrows():
            for _ in range(int(sentence_weight)):
                self.data.append((row['modern'], row['old_french'], 'sentence'))
        for _, row in words_df.iterrows():
            self.data.append((row['modern'], row['old_french'], 'word'))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        modern, old_french, typ = self.data[idx]
        if typ == 'sentence':
            input_text = f"translate French sentence to Old French: {modern}"
        else:
            input_text = f"translate French word to Old French: {modern}"

        inputs = self.tokenizer(
            input_text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True
        )
        targets = self.tokenizer(
            old_french,
            max_length=self.max_length,
            padding='max_length',
            truncation=True
        )
        return {
            'input_ids': torch.tensor(inputs['input_ids']),
            'attention_mask': torch.tensor(inputs['attention_mask']),
            'labels': torch.tensor(targets['input_ids'])
        }


In [9]:
# Datasets d'entraînement avec pondération
train_dataset = StratifiedOldFrenchDataset(
    sentences_df=train_sentences,
    words_df=train_words,
    # tokenizer=tokenizer,
    sentence_weight=3.0  # Les phrases sont 3x plus importantes
)

val_dataset = StratifiedOldFrenchDataset(
    sentences_df=val_sentences,
    words_df=val_words,
    # tokenizer=tokenizer,
    sentence_weight=3.0
)



In [10]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./mt5-old-french-stratified',
    num_train_epochs=4,
    per_device_train_batch_size=2,        # Très petit pour éviter OOM
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,       # Simule batch_size=16
    learning_rate=3e-4,
    warmup_steps=1000,
    weight_decay=0.01,
    logging_steps=200,
    eval_strategy='steps',
    eval_steps=1000,
    save_strategy='steps',
    save_steps=2000,
    fp16=True,
    dataloader_pin_memory=False,
    remove_unused_columns=False,
    # predict_with_generate=True,
    load_best_model_at_end=True,
    metric_for_best_model='eval_bleu',
)


In [11]:
# Métriques d'évaluation spécialisées - BLEU + métriques personnalisées pour vieux français

from sacrebleu import corpus_bleu
import numpy as np

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    
    # Décoder les prédictions
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # BLEU global
    bleu = corpus_bleu(decoded_preds, [decoded_labels])
    
    # Métriques séparées pour mots vs phrases
    word_preds = [p for p, l in zip(decoded_preds, decoded_labels) if len(l.split()) <= 2]
    word_labels = [l for l in decoded_labels if len(l.split()) <= 2]
    
    sentence_preds = [p for p, l in zip(decoded_preds, decoded_labels) if len(l.split()) > 2]
    sentence_labels = [l for l in decoded_labels if len(l.split()) > 2]
    
    # BLEU séparé
    word_bleu = corpus_bleu(word_preds, [word_labels]).score if word_preds else 0
    sentence_bleu = corpus_bleu(sentence_preds, [sentence_labels]).score if sentence_preds else 0
    
    return {
        'bleu': bleu.score,
        'word_bleu': word_bleu,
        'sentence_bleu': sentence_bleu,
        'word_samples': len(word_preds),
        'sentence_samples': len(sentence_preds)
    }



In [12]:
from transformers import DataCollatorForSeq2Seq

# Assurez-vous que 'tokenizer' et 'model' sont déjà définis
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)


In [13]:
# Entraînement avec Trainer - Lancement de l'entraînement

from transformers import Trainer
import torch

# Libérer la mémoire GPU
torch.cuda.empty_cache()

# Entraînement
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)


# Lancer l'entraînement
trainer.train()


/home/malek/BRIEFS DEV IA/14.NLP/EULA-vaaag-/.venv/lib/python3.11/site-packages/transformers/data/data_collator.py:741: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Step,Training Loss,Validation Loss


ValueError: You need to specify either `text` or `text_target`.